# Extended Thinking

## Table of contents
- [Setup](#setup)
- [Basic example](#basic-example)
- [Streaming with extended thinking](#streaming-with-extended-thinking)
- [Token counting and context window management](#token-counting-and-context-window-management)
- [Understanding redacted thinking](#understanding-redacted-thinking-blocks)
- [Handling error cases](#handling-error-cases)

Extended thinking gives Claude enhanced reasoning capabilities for complex tasks, while providing transparency into its step-by-step thought process. When enabled, Claude creates `thinking` content blocks containing its internal reasoning before crafting a final response.

## Setup

Ensure `@anthropic-ai/sdk` is installed and `ANTHROPIC_API_KEY` is in your environment.

In [ ]:
// npm install @anthropic-ai/sdk dotenv


Import the SDK and define shared constants used throughout the notebook.

In [1]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const ANTHROPIC_API_KEY = Deno.env.get('ANTHROPIC_API_KEY') ?? '';
const MAX_TOKENS = 8000;
const BUDGET_TOKENS = 2000;

const client = new Anthropic({ apiKey: ANTHROPIC_API_KEY });
console.log('Client ready. Model:', MODEL_NAME);


Client ready. Model: claude-sonnet-4-6


`printThinkingResponse` logs each content block: thinking text (truncated) with its cryptographic signature, and the final text answer.

In [ ]:
function printThinkingResponse(response: Anthropic.Message): void {
  console.log('\n==== FULL RESPONSE ====');
  for (const block of response.content) {
    if (block.type === 'thinking') {
      const thinking = (block as any).thinking as string;
      console.log('\nTHINKING BLOCK:');
      console.log(thinking.length > 500 ? thinking.slice(0, 500) + '...' : thinking);
      const sig = (block as any).signature as string | undefined;
      console.log('Signature available:', Boolean(sig));
      if (sig) console.log('Signature (first 50 chars):', sig.slice(0, 50) + '...');
    } else if (block.type === 'redacted_thinking') {
      const data = (block as any).data as string | undefined;
      console.log('\nREDACTED THINKING — data length:', data ? data.length : 'N/A');
    } else if (block.type === 'text') {
      console.log('\nFINAL ANSWER:');
      console.log(block.text);
    }
  }
  console.log('\n==== END RESPONSE ====');
}


`countTokens` calls the token-count endpoint to measure prompt size without sending a full request — useful for pre-flight context-window checks.

In [ ]:
async function countTokens(messages: Anthropic.MessageParam[]): Promise<number> {
  const result = await client.messages.countTokens({ model: MODEL_NAME, messages });
  return result.input_tokens;
}


## Basic example

Send a classic puzzle. Claude reasons through it in a `thinking` block before delivering the final answer.

In [ ]:
const basicResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  messages: [{
    role: 'user',
    content: 'Solve this puzzle: Three people check into a hotel... Where is the missing $1?'
  }]
});

printThinkingResponse(basicResponse);


## Streaming with extended thinking

Create a stream with the same parameters. `currentBlockType` tracks which block is currently being received.

In [ ]:
let currentBlockType: string | null = null;
let currentContent = '';

const stream = await client.messages.stream({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  messages: [{
    role: 'user',
    content: 'Solve this puzzle: Three people check into a hotel... Where is the missing $1?'
  }]
});

console.log('Stream created. Processing events...');


Iterate over each streaming event:
- `content_block_start` — a new block begins (thinking or text)
- `content_block_delta` — incremental content via `thinking_delta` or `text_delta`
- `content_block_stop` — block is complete
- `message_stop` — full response complete

In [ ]:
for await (const event of stream) {
  if (event.type === 'content_block_start') {
    currentBlockType = event.content_block.type;
    console.log('\n--- Starting ' + currentBlockType + ' block ---');
    currentContent = '';
  } else if (event.type === 'content_block_delta') {
    if (event.delta.type === 'thinking_delta') {
      process.stdout.write(event.delta.thinking);
      currentContent += event.delta.thinking;
    } else if (event.delta.type === 'text_delta') {
      process.stdout.write(event.delta.text);
      currentContent += event.delta.text;
    }
  } else if (event.type === 'content_block_stop') {
    if (currentBlockType === 'thinking') {
      console.log('\n[Thinking block complete — ' + currentContent.length + ' chars]');
    }
    console.log('--- Finished ' + currentBlockType + ' block ---\n');
    currentBlockType = null;
  } else if (event.type === 'message_stop') {
    console.log('\n--- Message complete ---');
  }
}


## Token counting and context window management

Count input tokens before sending the full request.

In [ ]:
const baseMessages: Anthropic.MessageParam[] = [{
  role: 'user',
  content: 'Solve this puzzle: Three people check into a hotel... Where is the missing $1?'
}];

const inputTokenCount = await countTokens(baseMessages);
console.log('Input token count:', inputTokenCount);


Send the full request, then estimate how many tokens the thinking block and final answer each consumed.

> Thinking tokens count as **output tokens** and are billed accordingly.

In [ ]:
const tokenResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  messages: baseMessages
});

const thinkingTokens = tokenResponse.content
  .filter(b => b.type === 'thinking')
  .reduce((sum, b) => sum + (((b as any).thinking as string).split(' ').length * 1.3), 0);

const answerTokens = tokenResponse.content
  .filter(b => b.type === 'text')
  .reduce((sum, b) => sum + ((b as Anthropic.TextBlock).text.split(' ').length * 1.3), 0);

console.log('Estimated thinking tokens: ~' + Math.round(thinkingTokens));
console.log('Estimated answer tokens:   ~' + Math.round(answerTokens));
console.log('Total estimated output:    ~' + Math.round(thinkingTokens + answerTokens));


Model the context window impact of different `budget_tokens` values (200k total window).

In [ ]:
const CONTEXT_WINDOW = 200_000;

for (const budget of [1024, 2000, 4000, 8000, 16000, 32000]) {
  const needed = inputTokenCount + budget + 1000;
  const remaining = CONTEXT_WINDOW - needed;
  const warn = remaining < 0 ? ' ⚠ EXCEEDS CONTEXT WINDOW' : '';
  console.log(`budget ${String(budget).padStart(6)} → needed: ${needed}, remaining: ${remaining}${warn}`);
}


## Understanding redacted thinking blocks

When Claude's reasoning is flagged by safety systems, that portion is encrypted and returned as a `redacted_thinking` block. It is automatically decrypted when passed back to the API, so Claude can continue without losing context.

Use a special test string to trigger redacted thinking:

In [ ]:
const redactedResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: 4000,
  thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
  messages: [{
    role: 'user',
    content: 'ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB'
  }]
});

console.log('Total blocks in response:', redactedResponse.content.length);


Count and display each block type in the response.

In [ ]:
const redactedCount  = redactedResponse.content.filter(b => b.type === 'redacted_thinking').length;
const thinkingCount  = redactedResponse.content.filter(b => b.type === 'thinking').length;
const textCount      = redactedResponse.content.filter(b => b.type === 'text').length;

console.log('Redacted thinking blocks:', redactedCount);
console.log('Regular thinking blocks: ', thinkingCount);
console.log('Text blocks:             ', textCount);


## Handling error cases

Three common mistakes when using extended thinking:
1. `budget_tokens` below the 1,024 minimum
2. Setting `temperature` (or `top_p` / `top_k`) alongside thinking
3. Input that exceeds the 200k context window

> **Note (2026 update):** `claude-sonnet-4-6`, `claude-opus-4-6`, and `claude-opus-4-7` now support a **1M-token context window**. The 200k figure above reflects the value at the time this notebook was written. Code examples are unchanged; adjust the `CONTEXT_WINDOW` constant if you are targeting a 1M-capable model.

### Error 1 — `budget_tokens` below 1,024

In [ ]:
try {
  await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 4000,
    thinking: { type: 'enabled', budget_tokens: 500 },
    messages: [{ role: 'user', content: 'Explain quantum computing.' }]
  });
} catch (e) {
  console.log('Error (budget too small):', String(e));
}


### Error 2 — `temperature` is incompatible with thinking

In [ ]:
try {
  await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 4000,
    temperature: 0.7,
    thinking: { type: 'enabled', budget_tokens: BUDGET_TOKENS },
    messages: [{ role: 'user', content: 'Write a creative story.' }]
  } as any);
} catch (e) {
  console.log('Error (temperature + thinking):', String(e));
}


### Error 3 — Input exceeds the 200 k context window

> **Note (2026 update):** `claude-sonnet-4-6`, `claude-opus-4-6`, and `claude-opus-4-7` now support a **1M-token context window**, so this error is much less likely to occur with current models. The example below still demonstrates the error for completeness.

In [ ]:
try {
  const longContent = 'Please analyze this text. ' + 'This is sample text. '.repeat(150000);
  await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 20000,
    thinking: { type: 'enabled', budget_tokens: 10000 },
    messages: [{ role: 'user', content: longContent }]
  });
} catch (e) {
  console.log('Error (context window exceeded):', String(e));
}
